# CNN-Based Semiconductor Wafer Defect Detection â€” Colab Quickstart

**AI 570 â€” Team 4** (Paul, Pyle, Rajan, Rettura)

End-to-end training on the WM-811K dataset with a free Colab T4 GPU. Wall-clock: ~20 minutes for all three models at 20 epochs.

## Before you start

1. **Enable GPU.** Menu bar â†’ `Runtime` â†’ `Change runtime type` â†’ `Hardware accelerator: T4 GPU` â†’ `Save`.
2. **Get the dataset.** You need `LSWMD_new.pkl` (~2.2 GB). Pick one option (do this BEFORE running cell 4 below):
   - **Option A (recommended):** upload `LSWMD_new.pkl` to your Google Drive at `MyDrive/datasets/LSWMD_new.pkl`
   - **Option B:** use the Colab upload dialog inside the notebook (slow â€” up to ~15 min over home broadband)
   - **Option C:** download from Kaggle inside Colab (needs `kaggle.json` API token â€” instructions below)

Then just run each cell top-to-bottom. The dataset cell auto-detects whichever option you chose.

## Cell 1. Clone repo and install

In [ ]:
REPO_URL = 'https://github.com/bpyle02/CNN-Based-Semiconductor-Wafer-Defect-Detection-Using-the-WM-811K-Dataset.git'
REPO_DIR = 'CNN-Based-Semiconductor-Wafer-Defect-Detection-Using-the-WM-811K-Dataset'

import os
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL}
%cd {REPO_DIR}
!git pull --ff-only
!pip install -q -e '.[dev]'
print('\nInstall complete. Restart runtime NOW if prompted, then re-run from here.')

## Cell 2. Verify GPU

In [ ]:
import torch
print('torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
    print('Total VRAM:', round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1), 'GB')
    !nvidia-smi | head -15
else:
    raise RuntimeError('No GPU detected. Menu: Runtime -> Change runtime type -> T4 GPU, then re-run this cell.')

## Cell 2b. Environment snapshot

Prints everything that typically causes "works on my machine" drift: Python
version, torch/CUDA versions, key env vars, and which GPU the runtime gave
us. Paste this into any bug report.

In [ ]:
import os, sys, platform, torch, subprocess

print(f'Python:          {sys.version.split()[0]} ({platform.platform()})')
print(f'torch:           {torch.__version__}')
print(f'CUDA available:  {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'CUDA version:    {torch.version.cuda}')
    print(f'cuDNN version:   {torch.backends.cudnn.version()}')
    print(f'GPU count:       {torch.cuda.device_count()}')
    for _i in range(torch.cuda.device_count()):
        _p = torch.cuda.get_device_properties(_i)
        print(f'  [{_i}] {_p.name}  {_p.total_memory/1e9:.1f} GB')

for _var in ['CUDA_VISIBLE_DEVICES', 'PYTHONPATH', 'TORCH_HOME',
             'TRANSFORMERS_CACHE', 'HF_HOME', 'WANDB_MODE']:
    print(f'env {_var!r}: {os.environ.get(_var, "<unset>")}')

print()
try:
    _ = subprocess.check_output(['nvidia-smi', '--query-gpu=name,memory.used,memory.total,driver_version',
                                 '--format=csv'], text=True)
    print(_)
except Exception:
    print('nvidia-smi not available (CPU runtime?).')


## Cell 3. Get the dataset into `data/LSWMD_new.pkl`

This cell tries each option in order and stops as soon as one succeeds:

1. Already present at `data/LSWMD_new.pkl` (skip)
2. Google Drive `MyDrive/datasets/LSWMD_new.pkl` â€” copies it in
3. Colab upload dialog â€” prompts you to pick the file from your computer
4. Kaggle â€” uploads your `kaggle.json` and pulls from the public dataset

**Most students will use option 2 (Drive).** Put `LSWMD_new.pkl` anywhere in your Drive and edit the `DRIVE_DATASET_PATH` below to point at it.

In [ ]:
import os, shutil

DATASET_PATH = 'data/LSWMD_new.pkl'
DRIVE_DATASET_PATH = '/content/drive/MyDrive/datasets/LSWMD_new.pkl'

os.makedirs('data', exist_ok=True)

def dataset_ready(path=DATASET_PATH, min_bytes=1_000_000_000):
    return os.path.exists(path) and os.path.getsize(path) > min_bytes

# Option 1: already present
if dataset_ready():
    print(f'Dataset already at {DATASET_PATH} ({os.path.getsize(DATASET_PATH)/1e9:.2f} GB)')

# Option 2: Google Drive
elif True:
    try:
        from google.colab import drive
        if not os.path.ismount('/content/drive'):
            drive.mount('/content/drive')
        if os.path.exists(DRIVE_DATASET_PATH):
            print(f'Copying from {DRIVE_DATASET_PATH} ...')
            shutil.copy(DRIVE_DATASET_PATH, DATASET_PATH)
            print(f'Done. Dataset size: {os.path.getsize(DATASET_PATH)/1e9:.2f} GB')
        else:
            print(f'NOT found at {DRIVE_DATASET_PATH}. Falling through to upload option.')
    except Exception as e:
        print(f'Drive mount/copy failed: {e}')

# Option 3: browser upload (only if still not ready)
if not dataset_ready():
    print('Opening upload dialog â€” select LSWMD_new.pkl from your computer (slow, ~10â€“15 min)...')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        name = list(uploaded.keys())[0]
        shutil.move(name, DATASET_PATH)
        print(f'Uploaded {name} -> {DATASET_PATH} ({os.path.getsize(DATASET_PATH)/1e9:.2f} GB)')

assert dataset_ready(), (
    'Dataset is not present. Options:\n'
    '  a) Put LSWMD_new.pkl at MyDrive/datasets/ in Google Drive and re-run this cell.\n'
    '  b) Use the upload dialog above.\n'
    '  c) See Cell 3b below for Kaggle download.'
)

### Cell 3b (optional). Kaggle download

Skip this if Cell 3 already loaded the dataset. To use this option:

1. Go to <https://www.kaggle.com/settings>, click **Create New Token** â€” this downloads `kaggle.json`.
2. Run this cell and upload `kaggle.json` when prompted.
3. The WM-811K public dataset will be downloaded and unpacked.

Note: the Kaggle download is the **raw** WM-811K (`LSWMD.pkl`, 1.9 GB), not the team's preprocessed `LSWMD_new.pkl`. The loader in `src/data/dataset.py` handles both.

In [ ]:
# import os
# from google.colab import files
# uploaded = files.upload()   # upload kaggle.json
# os.makedirs(os.path.expanduser('~/.kaggle'), exist_ok=True)
# os.rename('kaggle.json', os.path.expanduser('~/.kaggle/kaggle.json'))
# os.chmod(os.path.expanduser('~/.kaggle/kaggle.json'), 0o600)
# !pip install -q kaggle
# !kaggle datasets download -d qingyi/wm811k-wafer-map -p data/ --unzip
# # The file unpacks as data/LSWMD.pkl; rename if needed:
# import shutil
# if os.path.exists('data/LSWMD.pkl') and not os.path.exists('data/LSWMD_new.pkl'):
#     shutil.copy('data/LSWMD.pkl', 'data/LSWMD_new.pkl')
# print('size:', round(os.path.getsize('data/LSWMD_new.pkl')/1e9, 2), 'GB')

## Cell 4. Sanity-check the dataset

In [ ]:
%%time\n# Run the sanity check in a subprocess so the 4-5 GB dataset never stays in
# the Colab kernel's RAM. If this cell loaded `df` into the kernel, the
# training subprocess in Cell 6 would also load its own copy and the free
# tier's 12.7 GB RAM would blow. Keeping the kernel lean is essential.
!python -c "from src.data import load_dataset; df = load_dataset('data/LSWMD_new.pkl'); print('shape:', df.shape); print('columns:', df.columns.tolist()); print('\nFailure class distribution:'); print(df['failureClass'].value_counts().to_string())"

## Cell 5. Run the test suite (~1 min on T4)

In [ ]:
%%time\n!pytest -q --maxfail=5

## Cell 5b. Pre-resize the dataset once (~1â€“3 min on Colab)

The raw wafer maps have variable shapes (~26Ã—26 up to ~300Ã—300). The default training pipeline resizes each map on the fly with `skimage.transform.resize`, which is pure-Python and the biggest single CPU bottleneck on Colab â€” without this step the T4 is starved for data and per-epoch time balloons to ~10+ min.

This cell runs a one-time pre-resize using OpenCV + multiprocessing, caches the result to `data/LSWMD_cache.npz`, and `train.py` auto-detects the cache on subsequent runs. Expected output: `Resize complete in ~20â€“60 s (N maps/s). Cache shape: (172950, 96, 96), dtype: float16`.

In [ ]:
%%time\n!python scripts/precompute_tensors.py

## Cell 5c. 1-epoch preflight (~2 min)

Trains the custom CNN for a single epoch at a small batch size. Verifies
end-to-end wiring (data -> model -> loss -> optimizer -> checkpoint) before
committing to the full multi-model, multi-epoch run in Cell 6. Saves ~90
minutes if something is broken.

In [ ]:
%%time
# 1-epoch smoke test. If this fails, Cell 6 will definitely fail --
# fix the error here first.
!python train.py --model cnn --epochs 1 --batch-size 32 --device cuda --seed 42 2>&1 | tail -30


## Cell 6. Train all three models on GPU

Expected wall-clock on T4 at batch size 64, 10 epochs: ~15â€“20 minutes total.

The data loader uses 2 worker processes (Colab T4 has 2 vCPUs) and pinned memory, so per-epoch time on CNN/ResNet-18/EfficientNet-B0 is roughly 30â€“60 s.

On Colab free tier (12.7 GB RAM) this cell trains each model in a separate subprocess so RAM is released between models. If you have Colab Pro you can bump `BATCH_SIZE` to 128 and `EPOCHS` to 20.

If you hit `CUDA out of memory`, drop `BATCH_SIZE` to 32.

In [ ]:
%%time
EPOCHS = 10       # 10 is plenty for the baselines; bump to 20 once you've confirmed your setup is fast
BATCH_SIZE = 64   # T4-safe; Colab free tier has 12.7 GB RAM. Bump to 128 on Pro.
SEED = 42         # fixed seed -> bit-for-bit reproducible metrics for the report
USE_DRIVE_CKPT = False  # set True after Drive is mounted (see Cell 3 / Cell 8); uses configs/colab_fast.yaml

import os, gc, subprocess, sys, time, torch
from pathlib import Path

os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

# Aggressively release any lingering references in the kernel. Cell 4 now
# runs in a subprocess so df shouldn't be here, but IPython's output cache
# (_, __, ___, _oh) can still pin things. Reset those explicitly.
get_ipython().run_line_magic('reset_selective', '-f \"^_\"')
for _name in list(globals()):
    if _name.startswith('_') or _name in ('df', 'models', 'ensemble', 'gradcam'):
        globals().pop(_name, None)
for _ in range(3):
    gc.collect()
torch.cuda.empty_cache()

# RAM + GPU baseline
import psutil
_vm = psutil.virtual_memory()
print(f'Kernel RAM before training: {_vm.used/1e9:.1f} / {_vm.total/1e9:.1f} GB used')
if torch.cuda.is_available():
    _gpu = torch.cuda.get_device_properties(0)
    print(f'GPU: {_gpu.name} ({_gpu.total_memory/1e9:.1f} GB)')
else:
    print('WARNING: CUDA not available -- training will fall back to CPU (~2-3 hrs/model).')

Path('results').mkdir(exist_ok=True)

_failures = []
_t_overall = time.time()
for _model in ['cnn', 'resnet', 'efficientnet']:
    print(f'\n===== training {_model} =====', flush=True)
    _t0 = time.time()
    _cmd = [sys.executable, 'train.py', '--model', _model,
            '--epochs', str(EPOCHS), '--batch-size', str(BATCH_SIZE),
            '--device', 'cuda', '--seed', str(SEED)]
    if USE_DRIVE_CKPT:
        _cmd += ['--config', 'config.yaml', '--config', 'configs/colab_fast.yaml']
    _log_path = f'results/train_{_model}.log'
    # tee-style: stream subprocess output to both the notebook and a log file.
    # A Colab disconnect / scroll-off no longer loses the training history.
    with open(_log_path, 'w') as _logf:
        _proc = subprocess.Popen(_cmd, stdout=subprocess.PIPE,
                                  stderr=subprocess.STDOUT, bufsize=1,
                                  universal_newlines=True)
        for _line in _proc.stdout:
            sys.stdout.write(_line)
            sys.stdout.flush()
            _logf.write(_line)
        _rc = _proc.wait()
    _elapsed = time.time() - _t0
    if _rc == 0:
        print(f'\n[{_model}] OK ({_elapsed/60:.1f} min) -- full log: {_log_path}')
    else:
        print(f'\n[{_model}] FAILED with exit code {_rc} after {_elapsed/60:.1f} min')
        print(f'         log: {_log_path}')
        _failures.append(_model)
    for _ in range(3):
        gc.collect()
    torch.cuda.empty_cache()

print(f'\nTotal wall-clock: {(time.time()-_t_overall)/60:.1f} min')
if _failures:
    print(f'\n!!! FAILED MODELS: {_failures} -- scroll up or cat results/train_<model>.log')
    print('!!! Skip downstream cells (ensemble, Grad-CAM) until these are resolved.')
else:
    print('\nAll three models trained successfully. Proceed to evaluation cells.')


## Cell 7. Inspect results

In [ ]:
import os, json, glob

metric_files = sorted(glob.glob('results/*metrics.json'))
if not metric_files:
    print('No metrics.json files found in results/. Did training complete?')
for path in metric_files:
    with open(path) as f:
        m = json.load(f)
    acc = m.get('accuracy', m.get('test_accuracy', 0))
    macro = m.get('macro_f1', m.get('test_macro_f1', 0))
    weighted = m.get('weighted_f1', m.get('test_weighted_f1', 0))
    print(f'{os.path.basename(path):40s}  acc={acc:.4f}  macro_f1={macro:.4f}  weighted_f1={weighted:.4f}')

In [ ]:
from IPython.display import Image, display
import glob
imgs = sorted(glob.glob('results/*_confusion_matrix.png') + glob.glob('results/*_training_curves.png'))
if not imgs:
    print('No result images found.')
for path in imgs:
    print(path)
    display(Image(path))

## Cell 8. Save checkpoints and results to Drive

Colab free sessions disconnect after idle/disconnect events. Run this before closing the tab.

In [ ]:
import os, shutil, time

if not os.path.ismount('/content/drive'):
    from google.colab import drive
    drive.mount('/content/drive')

out_dir = f"/content/drive/MyDrive/wafer_runs/{time.strftime('%Y%m%d_%H%M%S')}"
os.makedirs(out_dir, exist_ok=True)

for src_dir in ['checkpoints', 'results']:
    if os.path.isdir(src_dir):
        shutil.copytree(src_dir, os.path.join(out_dir, src_dir), dirs_exist_ok=True)
print('Saved checkpoints and results to:', out_dir)

## Cell 9. Rare-class study (~45 min on T4, ~12 Pro units)

Runs the custom CNN under five rebalancing interventions
(baseline, focal loss, weighted CE + DRW, class-balanced sampling,
synthetic augmentation) x three seeds = 15 runs. Results and per-class
recall on rare classes (Near-full, Scratch, Donut, Random) are written
to `docs/rare_class_study.md`.

Uses the default `--epochs 10 --batch-size 64` to stay under the Colab
Pro compute budget. Bump to `--epochs 20 --batch-size 128` if you have
headroom. Resumable: skip already-completed (condition, seed) pairs
on rerun.

In [ ]:
%%time
!python scripts/run_rare_class_study.py \
    --epochs 10 --batch-size 64 --device cuda \
    --seeds 0,1,2 --conditions all

# Show the generated table so the run result is visible inline.
from IPython.display import Markdown, display
from pathlib import Path
_path = Path('docs/rare_class_study.md')
if _path.exists():
    display(Markdown(_path.read_text(encoding='utf-8')))
else:
    print('Study did not complete; docs/rare_class_study.md was not written.')


## Cell 10. SimCLR bonus (optional, ~20 min on T4)

Runs only after Cell 9 has written `docs/rare_class_study.md`. Pretrains
a CNN backbone self-supervised with SimCLR, then fine-tunes with standard
CE across the same 3 seeds, and appends a `F_simclr` section to the study
document.

Skip this cell if you're short on compute units. The core five-condition
study result in Cell 9 is the canonical deliverable.

In [ ]:
%%time
!python scripts/run_simclr_bonus.py \
    --pretrain-epochs 5 --finetune-epochs 10 \
    --batch-size 64 --device cuda \
    --seeds 0,1,2

from IPython.display import Markdown, display
from pathlib import Path
_path = Path('docs/rare_class_study.md')
if _path.exists():
    display(Markdown(_path.read_text(encoding='utf-8')))


---

**Done.** Inspect the confusion matrices and per-class F1 scores â€” macro-F1 is the imbalance-aware metric to report; accuracy is dominated by the 85% `none` class.